## DS6015-006 
## Derrick Clarke (thq3hn)

### Local Maxima Detection with scipy.ndimage to Identify individual Tree Crown Peaks

#### Conceptual Basis

<p>
Once the CHM is generated, individual tree tops appear as local maxima — pixels that are higher than all surrounding pixels within a defined search radius. The scipy.ndimage library provides a morphological maximum filter that efficiently identifies these peaks across the entire raster.
The algorithm works as follows:

    Apply a maximum filter with a circular footprint of radius r pixels to the CHM. This replaces each pixel's value with the maximum value found within the search window.
    Compare the filtered result to the original CHM. A pixel is a local maximum (i.e., a tree top) if and only if its original value equals the filtered value — meaning no neighbor within radius r is taller.
    Apply a minimum height threshold to exclude ground-level noise and low shrubs.
    Convert the pixel row/column indices of each detected peak to real-world coordinates (easting/northing), then reproject to latitude/longitude (WGS84).

</p>

### Imports

In [1]:
import numpy as np
import rasterio
from rasterio.transform import xy as rasterio_xy
from scipy.ndimage import maximum_filter, label
from pyproj import Transformer
import pandas as pd

In [2]:

# ── Step 1: Load the CHM ──────────────────────────────────────────────────────
with rasterio.open("../../data/outputs/GeoTIFF_files/chm_output.tif") as src:
    chm = src.read(1).astype(np.float32)
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

# Replace nodata with 0 for processing
if nodata is not None:
    chm = np.where(chm == nodata, 0, chm)
chm = np.nan_to_num(chm, nan=0.0)

print(f"CHM loaded: {chm.shape[0]} rows x {chm.shape[1]} cols")

# ── Step 2: Define the search window (crown radius) ──────────────────────────
# A 3 m search radius is appropriate for Central Virginia hardwood species.
# At 1 m/pixel resolution, this is a 7x7 pixel window (radius = 3 pixels).
# Adjust based on dominant species and resolution.

CROWN_RADIUS_M = 3.0          # meters — tune for your forest type
PIXEL_SIZE_M   = abs(transform.a)  # pixel size from raster metadata
search_radius_px = int(np.ceil(CROWN_RADIUS_M / PIXEL_SIZE_M))

# Build a circular footprint (disk-shaped structuring element)
y_idx, x_idx = np.ogrid[
    -search_radius_px : search_radius_px + 1,
    -search_radius_px : search_radius_px + 1
]
footprint = (x_idx**2 + y_idx**2) <= search_radius_px**2

print(f"Search radius: {CROWN_RADIUS_M} m = {search_radius_px} pixels")
print(f"Footprint size: {footprint.shape[0]} x {footprint.shape[1]} pixels")

# ── Step 3: Apply the maximum filter ─────────────────────────────────────────
chm_max_filtered = maximum_filter(chm, footprint=footprint)

# ── Step 4: Identify local maxima ────────────────────────────────────────────
# A pixel is a local maximum if it equals the filtered value (no taller neighbor)
local_maxima = (chm == chm_max_filtered)

# ── Step 5: Apply minimum height threshold ───────────────────────────────────
# Exclude anything below 2 m (shrubs, low vegetation, noise)
MIN_HEIGHT_M = 2.0
local_maxima = local_maxima & (chm >= MIN_HEIGHT_M)

print(f"Candidate tree tops detected: {local_maxima.sum():,}")

# ── Step 6: Extract row/column indices of each tree top ──────────────────────
rows, cols = np.where(local_maxima)

# ── Step 7: Convert pixel indices to projected coordinates (easting/northing) ─
# rasterio_xy returns (x, y) = (easting, northing) for each (row, col) pair
eastings, northings = rasterio_xy(transform, rows, cols)
eastings  = np.array(eastings)
northings = np.array(northings)

# ── Step 8: Reproject from UTM Zone 18N to WGS84 lat/lon ─────────────────────
# EPSG:32618 = UTM Zone 18N (standard for Central Virginia)
# EPSG:4326  = WGS84 geographic coordinates (lat/lon)
transformer = Transformer.from_crs("EPSG:32618", "EPSG:4326", always_xy=True)
longitudes, latitudes = transformer.transform(eastings, northings)

# ── Step 9: Extract CHM height at each tree top ───────────────────────────────
heights = chm[rows, cols]

# ── Step 10: Build a results DataFrame ───────────────────────────────────────
tree_tops = pd.DataFrame({
    "tree_id":   np.arange(1, len(rows) + 1),
    "latitude":  np.round(latitudes,  7),
    "longitude": np.round(longitudes, 7),
    "easting_m": np.round(eastings,   2),
    "northing_m":np.round(northings,  2),
    "height_m":  np.round(heights,    2),
    "row_px":    rows,
    "col_px":    cols
})

print(f"\nTree tops detected: {len(tree_tops):,}")
print(f"Height range: {tree_tops.height_m.min():.1f} m – {tree_tops.height_m.max():.1f} m")
print(tree_tops.head(10).to_string(index=False))

# ── Step 11: Save to CSV ──────────────────────────────────────────────────────
tree_tops.to_csv("../../data/outputs/tree_canopy_centroids.csv", index=False)
print("\nSaved: ../../data/outputs/tree_canopy_centroids.csv")


CHM loaded: 1500 rows x 1500 cols
Search radius: 3.0 m = 3 pixels
Footprint size: 7 x 7 pixels
Candidate tree tops detected: 8

Tree tops detected: 8
Height range: 2.0 m – 37.2 m
 tree_id  latitude  longitude  easting_m  northing_m  height_m  row_px  col_px
       1 37.892887 -72.157163   749981.5  4197741.49  2.050000     258    1481
       2 37.890867 -72.157809   749931.5  4197515.49  5.730000     484    1431
       3 37.890324 -72.160730   749676.5  4197447.49  3.750000     552    1176
       4 37.885926 -72.157306   749992.5  4196968.49  5.490000    1031    1492
       5 37.884830 -72.158178   749919.5  4196844.49 36.750000    1155    1419
       6 37.884424 -72.158933   749854.5  4196797.49 37.169998    1202    1354
       7 37.882323 -72.157752   749965.5  4196567.49 11.560000    1432    1465
       8 37.881879 -72.157996   749945.5  4196517.49 18.750000    1482    1445

Saved: ../../data/outputs/tree_canopy_centroids.csv
